# Algo #1: Equal Grouping of Order Sizes per Belt (Asc/Desc)

In [ ]:
import pandas as pd
import numpy as np


def load_order_sizes(itemtypes_path="Data/order_itemtypes.csv",
                     quantities_path="Data/order_quantities.csv",
                     size_method="total_qty"):
    """Load order sizes from data files.
    size_method: 'total_qty' = sum of quantities per order, 'line_items' = count of line items."""
    item_df = pd.read_csv(itemtypes_path, header=None)
    qty_df = pd.read_csv(quantities_path, header=None)

    if size_method == "total_qty":
        sizes = qty_df.sum(axis=1, skipna=True).astype(int)
    elif size_method == "line_items":
        sizes = item_df.notna().sum(axis=1).astype(int)
    else:
        raise ValueError("size_method must be 'total_qty' or 'line_items'")

    order_nums = pd.Series(np.arange(1, len(sizes) + 1), name="order_num")
    return pd.DataFrame({"order_num": order_nums, "size": sizes})

In [ ]:
def partition_into_k_strict_groups_minimax(order_sizes_df, k=4, descending=False, strict_boundary=True):
    """Partition orders into k *size-banded* groups to minimize the max group sum.

    Constraints:
    - Orders are sorted by size; each group must be a contiguous slice in that sorted list.
    - Group ordering is enforced by sorting: when `descending=True`, Group 1 has the largest sizes.

    Boundary rule:
    - If `strict_boundary=True`: max(group i) < min(group i+1) (cannot split equal sizes).
    - If `strict_boundary=False`: max(group i) <= min(group i+1) (equal sizes may split).

    Objective:
    - Minimize the maximum group SUM (balanced workload).
    """
    df = order_sizes_df.sort_values(["size", "order_num"], ascending=[True, True]).reset_index(drop=True)
    a = df["size"].to_list()
    ids = df["order_num"].to_list()
    n = len(a)

    if n < k:
        raise ValueError(f"Need at least {k} orders to make {k} non-empty groups.")

    def boundary_ok(i):
        if i == 0 or i == n:
            return True
        if not strict_boundary:
            return True
        return a[i - 1] < a[i]

    INF = 10**18
    P = [0]
    for v in a:
        P.append(P[-1] + v)

    dp = [[INF] * (n + 1) for _ in range(k + 1)]
    prev = [[-1] * (n + 1) for _ in range(k + 1)]
    dp[0][0] = 0

    for g in range(1, k + 1):
        for i in range(1, n + 1):
            if not boundary_ok(i):
                continue
            for j in range(g - 1, i):
                if dp[g - 1][j] == INF:
                    continue
                if not boundary_ok(j):
                    continue
                group_sum = P[i] - P[j]
                cost = max(dp[g - 1][j], group_sum)
                if cost < dp[g][i]:
                    dp[g][i] = cost
                    prev[g][i] = j

    if dp[k][n] == INF:
        raise ValueError(
            "Impossible to form strict < groups (likely too many equal sizes). "
            "Try size_method='total_qty' vs 'line_items', or relax strictness."
        )

    cuts = []
    i = n
    for g in range(k, 0, -1):
        j = prev[g][i]
        cuts.append((j, i))
        i = j
    cuts.reverse()

    groups = []
    for gi, (j, i) in enumerate(cuts, start=1):
        group_order_nums = ids[j:i]
        group_sizes = a[j:i]
        groups.append({
            "group": gi,
            "orders": group_order_nums,
            "sizes": group_sizes,
            "sum": sum(group_sizes),
            "min_size": min(group_sizes),
            "max_size": max(group_sizes),
        })

    if descending:
        groups = list(reversed(groups))
        for idx, g in enumerate(groups, start=1):
            g["group"] = idx

    return groups


def partition_into_k_groups_lpt(order_sizes_df, k=4, sort_groups_desc=True):
    """Load-balance into k groups to reduce the max group sum.

    Uses LPT (largest processing time first): sort orders by size descending and
    repeatedly assign the next order to the group with the smallest current sum.
    """
    df = order_sizes_df.sort_values(["size", "order_num"], ascending=[False, True]).reset_index(drop=True)

    groups = [
        {"orders": [], "sizes": [], "sum": 0}
        for _ in range(k)
    ]

    for _, row in df.iterrows():
        # Choose the currently-lightest group (tie-break by group index for stability)
        gi = min(range(k), key=lambda idx: (groups[idx]["sum"], idx))
        groups[gi]["orders"].append(int(row["order_num"]))
        groups[gi]["sizes"].append(int(row["size"]))
        groups[gi]["sum"] += int(row["size"])

    # Pretty-print: sort within each group by size descending (keep orders aligned)
    for g in groups:
        zipped = sorted(zip(g["sizes"], g["orders"]), key=lambda t: (-t[0], t[1]))
        g["sizes"], g["orders"] = [s for s, _ in zipped], [o for _, o in zipped]
        g["min_size"] = min(g["sizes"]) if g["sizes"] else None
        g["max_size"] = max(g["sizes"]) if g["sizes"] else None

    if sort_groups_desc:
        groups = sorted(groups, key=lambda g: (-g["sum"], -g["max_size"], g["min_size"]))

    out = []
    for idx, g in enumerate(groups, start=1):
        out.append({
            "group": idx,
            "orders": g["orders"],
            "sizes": g["sizes"],
            "sum": g["sum"],
            "min_size": g["min_size"],
            "max_size": g["max_size"],
        })
    return out

# Algo #2: Equal Sum of Order Size per Belt Group

In [ ]:
df = load_order_sizes(size_method="total_qty")

print("Order sizes (sorted):", df.sort_values("size")["size"].tolist())

# ----------------------------
# A) Size-banded groups with non-strict adjacency:
#    Group 1 sizes >= Group 2 sizes >= ... (equal sizes may split across groups)
# ----------------------------
banded_groups = partition_into_k_strict_groups_minimax(
    df, k=4, descending=True, strict_boundary=False
)
print("\nSize-banded groups (>= across adjacent groups):")
for g in banded_groups:
    print(f"  Group {g['group']}: orders {g['orders']}, sizes {g['sizes']}, sum={g['sum']}")
print("Max group sum (banded):", max(g["sum"] for g in banded_groups))

# ----------------------------
# B) Load-balanced groups (mixing allowed; not size-banded)
# ----------------------------
lpt_groups = partition_into_k_groups_lpt(df, k=4, sort_groups_desc=True)
print("\nLoad-balanced groups (mixing allowed):")
for g in lpt_groups:
    print(f"  Group {g['group']}: orders {g['orders']}, sizes {g['sizes']}, sum={g['sum']}")
print("Max group sum (load-balanced):", max(g["sum"] for g in lpt_groups))

Order sizes (sorted): [1, 2, 2, 2, 3, 3, 3, 4, 5, 5, 6]

Size-banded groups (>= across adjacent groups):
  Group 1: orders [9], sizes [6], sum=6
  Group 2: orders [1, 2], sizes [5, 5], sum=10
  Group 3: orders [8, 10, 4], sizes [3, 3, 4], sum=10
  Group 4: orders [11, 3, 5, 6, 7], sizes [1, 2, 2, 2, 3], sum=10
Max group sum (banded): 10

Load-balanced groups (mixing allowed):
  Group 1: orders [9, 3, 6], sizes [6, 2, 2], sum=10
  Group 2: orders [1, 8, 11], sizes [5, 3, 1], sum=9
  Group 3: orders [4, 7, 5], sizes [4, 3, 2], sum=9
  Group 4: orders [2, 10], sizes [5, 3], sum=8
Max group sum (load-balanced): 10


# Algo #3: Order Grouping by Item Similarity

In [ ]:
def load_item_sets(itemtypes_path="Data/order_itemtypes.csv", size_method="line_items"):
    """Load item sets and sizes for each order.
    Returns a list of dictionaries with order_num, size, and item_set."""
    # Use the existing load_order_sizes to get the base dataframe with sizes
    df_sizes = load_order_sizes(itemtypes_path=itemtypes_path, size_method=size_method)

    # Load the item types directly for set conversion
    item_df = pd.read_csv(itemtypes_path, header=None)

    # Convert each row to a set of unique items as integers, dropping NaN values
    item_sets = item_df.apply(lambda row: set(row.dropna().astype(int)), axis=1).tolist()

    # Add item_sets to our dataframe
    df_sizes['item_set'] = item_sets

    # Convert to a list of records for easier iteration/processing in similarity algos
    return df_sizes.to_dict(orient='records')

# Example usage to verify
order_data = load_item_sets()
print(f"Loaded {len(order_data)} orders.")
if order_data:
    print(f"Sample Order 1: {order_data[0]}")

Loaded 11 orders.
Sample Order 1: {'order_num': 1, 'size': 2, 'item_set': {1, 3}}


In [ ]:
def jaccard_similarity(set1, set2):
    """Calculate Jaccard Similarity between two sets."""
    if not set1 or not set2:
        return 0.0
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union

def partition_into_k_similarity_groups(order_list, k=4):
    """Group orders to maximize item-type overlap using Jaccard Similarity."""
    # Sort by set size descending to pick diverse seeds
    sorted_orders = sorted(order_list, key=lambda x: len(x['item_set']), reverse=True)

    # Initialize k groups with the first k orders as seeds
    groups = []
    for i in range(k):
        order = sorted_orders[i]
        groups.append({
            "group": i + 1,
            "orders": [order['order_num']],
            "sizes": [order['size']],
            "item_sets": [order['item_set']],
            "sum": order['size']
        })

    # Assign remaining orders
    remaining_orders = sorted_orders[k:]
    for order in remaining_orders:
        best_score = -1.0
        best_group_idx = 0

        for idx, group in enumerate(groups):
            # Calculate average Jaccard similarity with existing orders in group
            scores = [jaccard_similarity(order['item_set'], s) for s in group['item_sets']]
            avg_score = sum(scores) / len(scores) if scores else 0

            if avg_score > best_score:
                best_score = avg_score
                best_group_idx = idx

        # Add to the most similar group
        target = groups[best_group_idx]
        target['orders'].append(order['order_num'])
        target['sizes'].append(order['size'])
        target['item_sets'].append(order['item_set'])
        target['sum'] += order['size']

    # Finalize format to match other algorithms
    for g in groups:
        g['min_size'] = min(g['sizes']) if g['sizes'] else 0
        g['max_size'] = max(g['sizes']) if g['sizes'] else 0
        # Union of all items in the group for summary
        g['unique_items'] = set().union(*g['item_sets'])
        del g['item_sets'] # Remove raw sets to clean up output

    return groups

# Run the similarity algorithm
similarity_groups = partition_into_k_similarity_groups(order_data, k=4)

print("Item Similarity Groups:")
for g in similarity_groups:
    print(f"  Group {g['group']}: orders {g['orders']}, sum={g['sum']}, unique_item_count={len(g['unique_items'])}")
print("Max group sum (Similarity):", max(g['sum'] for g in similarity_groups))

Item Similarity Groups:
  Group 1: orders [2, 8, 10], sum=6, unique_item_count=4
  Group 2: orders [9], sum=3, unique_item_count=3
  Group 3: orders [1, 5, 6, 11], sum=6, unique_item_count=3
  Group 4: orders [4, 7, 3], sum=5, unique_item_count=3
Max group sum (Similarity): 6


In [ ]:

print("# Algo #3: Item Similarity Results Summary")
for g in similarity_groups:
    print(f"Belt {g['group']}:")
    print(f"  - Orders: {g['orders']}")
    print(f"  - Total Size (Sum): {g['sum']}")
    print(f"  - Unique Item Types: {sorted(list(g['unique_items']))}")
    print(f"  - Item Type Count: {len(g['unique_items'])}")
    print("-" * 30)

print("\nFinal Confirmation:")
print("1. Algo #1: Equal Grouping (Banded) - Completed")
print("2. Algo #2: Equal Sum (Load Balanced) - Completed")
print("3. Algo #3: Item Similarity (Jaccard) - Completed")

# Algo #3: Item Similarity Results Summary
Belt 1:
  - Orders: [2, 8, 10]
  - Total Size (Sum): 6
  - Unique Item Types: [2, 3, 4, 6]
  - Item Type Count: 4
------------------------------
Belt 2:
  - Orders: [9]
  - Total Size (Sum): 3
  - Unique Item Types: [0, 1, 5]
  - Item Type Count: 3
------------------------------
Belt 3:
  - Orders: [1, 5, 6, 11]
  - Total Size (Sum): 6
  - Unique Item Types: [1, 2, 3]
  - Item Type Count: 3
------------------------------
Belt 4:
  - Orders: [4, 7, 3]
  - Total Size (Sum): 5
  - Unique Item Types: [0, 3, 5]
  - Item Type Count: 3
------------------------------

Final Confirmation:
1. Algo #1: Equal Grouping (Banded) - Completed
2. Algo #2: Equal Sum (Load Balanced) - Completed
3. Algo #3: Item Similarity (Jaccard) - Completed


# Algorithm CSV Outputs

### Helper Function: Mapping Orders to CSV Format

In [ ]:
def create_algo_csv(groups, filename):
    # Load raw item types and quantities to map back to shapes
    item_df = pd.read_csv("Data/order_itemtypes.csv", header=None)
    qty_df = pd.read_csv("Data/order_quantities.csv", header=None)

    # Shape mapping: circle:1, pentagon:2, trapezoid:3, triangle:4, star:5, moon:6, heart:7, cross:8
    shape_cols = ['circle', 'pentagon', 'trapezoid', 'triangle', 'star', 'moon', 'heart', 'cross']

    rows = []
    for g in groups:
        conv_idx = g['group'] - 1 # 0-indexed conveyor number
        # Initialize counts for this conveyor
        counts = {shape: 0 for shape in shape_cols}

        for order_num in g['orders']:
            # order_num is 1-indexed in the notebook logic
            row_idx = order_num - 1
            items = item_df.iloc[row_idx].dropna().astype(int).tolist()
            qties = qty_df.iloc[row_idx].dropna().astype(int).tolist()

            for item_id, qty in zip(items, qties):
                if 1 <= item_id <= 8:
                    shape_name = shape_cols[item_id - 1]
                    counts[shape_name] += qty

        row = {'conv_num': conv_idx}
        row.update(counts)
        rows.append(row)

    final_df = pd.DataFrame(rows)
    final_df.to_csv(filename, index=False)
    print(f"--- Summary for {filename} ---")
    print(final_df.to_string(index=False))
    return final_df

## Algorithm 1: Equal Grouping (Size-Banded)

In [231]:
import pandas as pd

# 1. Run Algo #1 logic
df_sizes = load_order_sizes(size_method="total_qty")
banded_groups = partition_into_k_strict_groups_minimax(df_sizes, k=4, descending=True, strict_boundary=False)

# 2. Map back to 11 individual rows
item_df = pd.read_csv("Data/order_itemtypes.csv", header=None)
qty_df = pd.read_csv("Data/order_quantities.csv", header=None)
shape_cols = ['circle', 'pentagon', 'trapezoid', 'triangle', 'star', 'moon', 'heart', 'cross']

order_rows = []
for g in banded_groups:
    for order_num in g['orders']:
        idx = order_num - 1
        row = {'conv_num': g['group']} # Changed from g['group'] - 1 to g['group']
        # Initialize shapes to 0
        for s in shape_cols: row[s] = 0
        # Fill quantities
        items = item_df.iloc[idx].dropna().astype(int).tolist()
        qties = qty_df.iloc[idx].dropna().astype(int).tolist()
        for i_id, q in zip(items, qties):
            if 1 <= i_id <= 8: row[shape_cols[i_id-1]] = q
        order_rows.append(row)

algo1_df = pd.DataFrame(order_rows)
algo1_df.to_csv("order-sequence-algo1.csv", index=False)
print("Preview: Algo 1 (Individual Orders)")
print(algo1_df)

FileNotFoundError: [Errno 2] No such file or directory: 'Data/order_itemtypes.csv'

## Algorithm 2: Equal Sum (Load Balanced)

In [ ]:
# 1. Run Algo #2 logic
df_sizes = load_order_sizes(size_method="total_qty")
lpt_groups = partition_into_k_groups_lpt(df_sizes, k=4, sort_groups_desc=True)

# 2. Map back to individual rows
order_rows = []
for g in lpt_groups:
    for order_num in g['orders']:
        idx = order_num - 1
        row = {'conv_num': g['group']} # Changed from g['group'] - 1 to g['group']
        for s in shape_cols: row[s] = 0
        items = item_df.iloc[idx].dropna().astype(int).tolist()
        qties = qty_df.iloc[idx].dropna().astype(int).tolist()
        for i_id, q in zip(items, qties):
            if 1 <= i_id <= 8: row[shape_cols[i_id-1]] = q
        order_rows.append(row)

algo2_df = pd.DataFrame(order_rows)
algo2_df.to_csv("order-sequence-algo2.csv", index=False)
print("Preview: Algo 2 (Individual Orders)")
print(algo2_df)

Preview: Algo 2 (Individual Orders)
    conv_num  circle  pentagon  trapezoid  triangle  star  moon  heart  cross
0          0       2         0          0         0     1     0      0      0
1          0       0         0          0         0     2     0      0      0
2          0       2         0          0         0     0     0      0      0
3          1       2         0          3         0     0     0      0      0
4          1       0         1          0         2     0     0      0      0
5          1       1         0          0         0     0     0      0      0
6          2       0         0          0         0     1     0      0      0
7          2       0         0          2         0     1     0      0      0
8          2       1         1          0         0     0     0      0      0
9          3       0         1          3         1     0     0      0      0
10         3       0         0          0         0     0     3      0      0


## Algorithm 3: Item Similarity

In [ ]:
# 1. Run Algo #3 logic
order_data = load_item_sets(size_method="line_items")
similarity_groups = partition_into_k_similarity_groups(order_data, k=4)

# 2. Map back to individual rows
order_rows = []
for g in similarity_groups:
    for order_num in g['orders']:
        idx = order_num - 1
        row = {'conv_num': g['group']} # Changed from g['group'] - 1 to g['group']
        for s in shape_cols: row[s] = 0
        items = item_df.iloc[idx].dropna().astype(int).tolist()
        qties = qty_df.iloc[idx].dropna().astype(int).tolist()
        for i_id, q in zip(items, qties):
            if 1 <= i_id <= 8: row[shape_cols[i_id-1]] = q
        order_rows.append(row)

algo3_df = pd.DataFrame(order_rows)
algo3_df.to_csv("order-sequence-algo3.csv", index=False)
print("Preview: Algo 3 (Individual Orders)")
print(algo3_df)

Preview: Algo 3 (Individual Orders)
    conv_num  circle  pentagon  trapezoid  triangle  star  moon  heart  cross
0          0       0         1          3         1     0     0      0      0
1          0       0         1          0         2     0     0      0      0
2          0       0         0          0         0     0     3      0      0
3          1       2         0          0         0     1     0      0      0
4          2       2         0          3         0     0     0      0      0
5          2       1         1          0         0     0     0      0      0
6          2       2         0          0         0     0     0      0      0
7          2       1         0          0         0     0     0      0      0
8          3       0         0          0         0     1     0      0      0
9          3       0         0          2         0     1     0      0      0
10         3       0         0          0         0     2     0      0      0
